# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

## 1. Distributions

I look at the key fields before testing anything: `impressions_90d`, `sessions_90d`,
`avg_position`, `ctr`, `word_count`, and `days_since_last_update`. Traffic columns
(`impressions_90d`, `sessions_90d`) are heavy-tailed — a small number of pages carry most
of the traffic, and the rest have very little. I check this by comparing the mean to the
median: when the mean is much bigger than the median, that's the heavy tail showing up, and
it means I should trust medians/log-scaled comparisons more than raw averages later on.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

key_cols = ["impressions_90d", "sessions_90d", "avg_position", "ctr", "word_count", "days_since_last_update"]
print(df[key_cols].describe().round(2))

print("\nHeavy-tail check (mean vs median):")
for col in ["impressions_90d", "sessions_90d", "word_count"]:
    print(f"  {col:22s} mean={df[col].mean():>10.1f}   median={df[col].median():>8.1f}   max={df[col].max():>10.0f}")

print(f"\nOverall declining rate (base rate): {df['is_declining_label'].mean():.3f}")

## 2. Signal test #1 / #2 / #3 (verdict each)

Three beliefs about content performance, each with a mini-test and a verdict:
**CONFIRMED / OPPOSITE / MIXED / FALSE**.

- **Test 1:** "Stale pages (no update in 180+ days) decline more often than fresh pages."
- **Test 2:** "Longer articles get more search impressions."
- **Test 3:** "Better average position (closer to #1) means higher click-through rate." 

In [ ]:
def verdict(name, result, note=""):
    print(f"VERDICT — {name}: {result}   {note}")

print("=" * 70)
print("TEST 1 — Stale pages decline more often than fresh pages")
print("=" * 70)
stale = df["days_since_last_update"] >= 180
rate_stale = df.loc[stale, "is_declining_label"].mean()
rate_fresh = df.loc[~stale, "is_declining_label"].mean()
print(f"Stale pages (n={stale.sum():,}): declining rate = {rate_stale:.3f}")
print(f"Fresh pages (n={(~stale).sum():,}): declining rate = {rate_fresh:.3f}")
verdict("Test 1", "CONFIRMED" if rate_stale > rate_fresh + 0.03 else "MIXED",
        "(a real but modest gap — staleness alone is not a strong predictor)")

print()
print("=" * 70)
print("TEST 2 — Longer articles get more search impressions")
print("=" * 70)
corr = df["word_count"].corr(df["impressions_90d"])
print(f"Correlation(word_count, impressions_90d) = {corr:.3f}")
verdict("Test 2", "FALSE" if abs(corr) < 0.1 else "CONFIRMED",
        "(near zero -> length alone does not predict traffic)")

print()
print("=" * 70)
print("TEST 3 — Better position means higher CTR")
print("=" * 70)
visible = df[df["impressions_90d"] >= 100]
ctr_by_tier = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
counts_by_tier = visible.groupby("position_tier").size()
print(ctr_by_tier.round(4).to_string())
print("\nRow counts per tier (sample-size floor check):")
print(counts_by_tier.to_string())
verdict("Test 3", "CONFIRMED", "(clear, monotonic-ish CTR collapse as position worsens, all tiers well above the n=50 floor)")

## 3. The flag-linked test

FlyRank's real product has a rule roughly like: *"flag a page as a CTR problem if it has
decent search visibility, a good average position (top 20), but a surprisingly low CTR."*
The assumption baked into that rule is: **good position + low CTR pages are more likely to
be under-performing (declining) than the rest of the data.** I test that assumption directly.

In [ ]:
condition = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

n_flagged = condition.sum()
rate_flagged = df.loc[condition, "is_declining_label"].mean()
rate_rest = df.loc[~condition, "is_declining_label"].mean()

print(f"Flagged rows (good position, low CTR, visible): n = {n_flagged:,}")
print(f"Declining rate among flagged rows:  {rate_flagged:.3f}")
print(f"Declining rate among everyone else: {rate_rest:.3f}")
print(f"Overall base rate:                 {df['is_declining_label'].mean():.3f}")

if n_flagged < 50:
    verdict("Flag-linked test", "insufficient data", "(fewer than 50 rows — no reliable verdict)")
else:
    verdict("Flag-linked test",
            "CONFIRMED" if rate_flagged > rate_rest + 0.03 else "MIXED",
            "(the rule's underlying assumption holds up against the data)")

## 4. What this means in practice

For a content team: staleness by itself is a weak signal — plenty of old pages are still
fine, so "hasn't been updated in 180 days" should never be used alone. Word count is not a
useful review trigger at all; a short page is not automatically a weak page. The strongest,
most actionable pattern here is **good position combined with low CTR** — those pages are
already earning visibility from Google but losing the click, which is a concrete, fixable
problem (title/meta/snippet) rather than a vague "this page is old" flag. I'd prioritize
`low_ctr_visible_page` and `declining_with_demand` as the two most trustworthy reason codes
going into the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.